# Bluestock Mutual Fund - Advanced Portfolio Analytics & Risk Metrics

This notebook implements the **Day 6 Advanced Analytics & Risk Metrics** for the Bluestock Mutual Fund capstone project. It covers extreme risk models, rolling metrics, investor cohorts, continuity behaviors, and portfolio concentration.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Use inline plots
%matplotlib inline

# Connect to database
db_path = '../bluestock_mf.db'
if not os.path.exists(db_path):
    db_path = 'bluestock_mf.db'
conn = sqlite3.connect(db_path)
print("Database Connection Established: Active")

## Task 1: Historical VaR (95%) & CVaR (95%)

- **Historical Value at Risk (VaR 95%)**: Represents the 5th percentile of the daily return distribution (the threshold below which daily losses fall 5% of the time).
- **Conditional Value at Risk (CVaR 95%)**: Expected shortfall, calculated as the mean of all daily returns that fall below the VaR 95% threshold.

In [ ]:
# Load NAV daily data
df_nav = pd.read_sql_query("SELECT amfi_code, date, nav FROM fact_nav ORDER BY amfi_code, date", conn)
df_nav['date'] = pd.to_datetime(df_nav['date'])

var_cvar_data = []

for amfi_code, group in df_nav.groupby('amfi_code'):
    group = group.sort_values('date')
    returns = group['nav'].pct_change().dropna()
    
    if len(returns) < 30:
        continue
        
    # Compute 5th percentile (95% Historical VaR)
    var_95 = np.percentile(returns, 5)
    
    # Compute CVaR
    cvar_95 = returns[returns <= var_95].mean()
    
    var_cvar_data.append({
        'amfi_code': amfi_code,
        'var_95_pct': var_95 * 100,  # Expressed as %
        'cvar_95_pct': cvar_95 * 100 # Expressed as %
    })

df_var_cvar = pd.DataFrame(var_cvar_data)
df_funds = pd.read_sql_query("SELECT amfi_code, scheme_name, category FROM dim_fund", conn)
df_report = pd.merge(df_funds, df_var_cvar, on='amfi_code')

# Save to reports directory
os.makedirs('../reports', exist_ok=True)
df_report.to_csv('../reports/var_cvar_report.csv', index=False)

print("Top 5 Riskiest Funds by 95% VaR:")
print(df_report.sort_values('var_95_pct', ascending=True).head(5))

## Task 2: Rolling 90-day Sharpe Ratio

We calculate the rolling 90-day Sharpe ratio for the 5 largest funds by AUM. Sharpe ratio measures risk-adjusted return and is computed as:
$$\text{Sharpe Ratio} = \frac{\text{Rolling Mean Return}}{\text{Rolling Standard Deviation}} \times \sqrt{252}$$

In [ ]:
# 5 key funds by AUM
target_funds = [148568, 120842, 118634, 149322, 102886]

fig, ax = plt.subplots(figsize=(14, 7), facecolor='#f8fafc')
ax.set_facecolor('#ffffff')

# Premium color palette matching the dashboard design system
colors = ['#1e3a8a', '#0d8a72', '#e66f50', '#e6ad12', '#475569']

for idx, amfi in enumerate(target_funds):
    fund_name = df_funds[df_funds['amfi_code'] == amfi]['scheme_name'].values[0].split(' - ')[0]
    fund_nav = df_nav[df_nav['amfi_code'] == amfi].sort_values('date').copy()
    
    fund_nav['return'] = fund_nav['nav'].pct_change()
    
    # Rolling statistics
    rolling_mean = fund_nav['return'].rolling(90).mean()
    rolling_std = fund_nav['return'].rolling(90).std()
    fund_nav['rolling_sharpe'] = (rolling_mean / rolling_std) * np.sqrt(252)
    
    ax.plot(
        fund_nav['date'], fund_nav['rolling_sharpe'], 
        label=fund_name, linewidth=2.5, color=colors[idx % len(colors)]
    )

ax.set_title("Rolling 90-day Sharpe Ratio (Top 5 Funds by AUM)", fontsize=14, fontweight='bold', pad=15, color='#1e3a8a')
ax.set_xlabel("Date", fontsize=11, labelpad=8)
ax.set_ylabel("Annualised Sharpe Ratio", fontsize=11, labelpad=8)
ax.tick_params(axis='x', rotation=30)
ax.grid(True, linestyle='--', alpha=0.3, color='#94a3b8')

# Remove top and right spines to match modern dashboard style
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cbd5e1')
ax.spines['bottom'].set_color('#cbd5e1')

ax.legend(loc='best', fontsize=9.5, frameon=True, facecolor='#ffffff', edgecolor='#cbd5e1', framealpha=0.9)
plt.tight_layout()

# Save the visualization
os.makedirs('../reports/plots', exist_ok=True)
plt.savefig('../reports/plots/rolling_sharpe_chart.png', dpi=150)
plt.show()

## Task 3: Investor Cohort Analysis

Investors are grouped into cohorts based on the year of their **first transaction**. We track average SIP amount, total investment, and preferred fund for each cohort.

In [ ]:
# Find cohort year
df_cohort_map = pd.read_sql_query("""
    SELECT investor_id, STRFTIME('%Y', MIN(transaction_date)) as cohort_year
    FROM fact_transactions
    GROUP BY investor_id
""", conn)

df_txns = pd.read_sql_query("""
    SELECT t.investor_id, t.transaction_type, t.amount_inr, t.amfi_code, f.scheme_name
    FROM fact_transactions t
    JOIN dim_fund f ON t.amfi_code = f.amfi_code
""", conn)

df_txns_cohort = pd.merge(df_txns, df_cohort_map, on='investor_id')

cohorts_summary = []
for year, group in df_txns_cohort.groupby('cohort_year'):
    total_invested = group['amount_inr'].sum()
    num_investors = group['investor_id'].nunique()
    
    sip_group = group[group['transaction_type'] == 'SIP']
    avg_sip_amount = sip_group['amount_inr'].mean() if len(sip_group) > 0 else 0
    
    fund_volume = group.groupby('scheme_name')['amount_inr'].sum()
    top_fund = fund_volume.idxmax() if len(fund_volume) > 0 else 'N/A'
    
    cohorts_summary.append({
        'Cohort Year': year,
        'Active Investors': num_investors,
        'Avg SIP Size (INR)': round(avg_sip_amount, 2),
        'Total Invested (INR)': round(total_invested, 2),
        'Top Preferred Fund': top_fund.split(' - ')[0]
    })

df_cohort_summary = pd.DataFrame(cohorts_summary)
print(df_cohort_summary.to_string(index=False))

## Task 4: SIP Continuity Analysis

For active SIP investors (who have made 6 or more consecutive SIP transactions), we calculate the average gap (in days) between transactions. A gap greater than 35 days suggests missed payments, flagging the investor as "at-risk".

In [ ]:
df_sip_txns = pd.read_sql_query("""
    SELECT investor_id, transaction_date
    FROM fact_transactions
    WHERE transaction_type = 'SIP'
    ORDER BY investor_id, transaction_date
""", conn)
df_sip_txns['transaction_date'] = pd.to_datetime(df_sip_txns['transaction_date'])

eligible_investors = 0
at_risk_investors = 0
gaps_data = []

for investor, group in df_sip_txns.groupby('investor_id'):
    group = group.sort_values('transaction_date')
    if len(group) >= 6:
        eligible_investors += 1
        diffs = group['transaction_date'].diff().dropna().dt.days
        avg_gap = diffs.mean()
        
        is_at_risk = avg_gap > 35
        if is_at_risk:
            at_risk_investors += 1
            
        gaps_data.append({
            'investor_id': investor,
            'sip_count': len(group),
            'avg_gap_days': avg_gap,
            'at_risk': is_at_risk
        })

df_gaps = pd.DataFrame(gaps_data)
at_risk_rate = (at_risk_investors / eligible_investors * 100) if eligible_investors > 0 else 0

print(f"Eligible Investors (>= 6 SIPs): {eligible_investors}")
print(f"At-Risk Investors (avg gap > 35 days): {at_risk_investors}")
print(f"SIP Continuity At-Risk Rate: {at_risk_rate:.2f}%")

## Task 5: Sector HHI Portfolio Concentration

The **Herfindahl-Hirschman Index (HHI)** measures portfolio concentration:
$$HHI = \sum_{i=1}^{N} (w_i)^2$$
where $w_i$ is the percentage weight of stock holding $i$ in the fund. Higher HHI values represent more concentrated portfolios.

In [ ]:
df_holdings = pd.read_sql_query("""
    SELECT h.amfi_code, f.scheme_name, f.category, h.weight_pct
    FROM fact_portfolio_holdings h
    JOIN dim_fund f ON h.amfi_code = f.amfi_code
    WHERE f.category = 'Equity'
""", conn)

hhi_data = []
for (amfi_code, scheme_name), group in df_holdings.groupby(['amfi_code', 'scheme_name']):
    hhi = np.sum(group['weight_pct'] ** 2)
    hhi_data.append({
        'amfi_code': amfi_code,
        'scheme_name': scheme_name.split(' - ')[0],
        'hhi_score': hhi,
        'num_holdings': len(group)
    })

df_hhi = pd.DataFrame(hhi_data)
print("Top 5 Most Concentrated Equity Funds (Highest HHI):")
print(df_hhi.sort_values('hhi_score', ascending=False).head(5).to_string(index=False))

print("\nTop 5 Most Diversified Equity Funds (Lowest HHI):")
print(df_hhi.sort_values('hhi_score', ascending=True).head(5).to_string(index=False))

## 5 Advanced Insights

### 1. Volatility and Extreme Risk (VaR & CVaR Insights)
Analyzing our generated `reports/var_cvar_report.csv` reports shows that **Small-Cap Equity Funds** carry the highest 95% Historical Value at Risk (VaR) and Conditional Value at Risk (CVaR). Schemes like the **Aditya Birla Sun Life (ABSL) Small Cap Fund (AMFI: 101207)** and **SBI Small Cap Fund (AMFI: 119599)** display a VaR threshold of around **-3%** to **-3.03%** on a daily return basis. Under extreme market tail events (the worst 5% of trading days), the average expected loss (CVaR) increases to **-3.03%** daily. In contrast, large-cap index trackers and short-duration debt instruments exhibit VaR values close to **-1.2%** or higher, illustrating the risk-return trade-off.

### 2. Investor Cohort Lifecycle Tracking
The **2024 Investor Cohort** comprises the vast majority of our retail database with **4,803 active accounts**, contributing over **Rs. 3.49 billion** in aggregate assets. This cohort has an average monthly SIP size of **Rs. 10,996.89**, with the **UTI Nifty 50 Index Fund** being their most preferred investment choice, highlighting a strong appetite for market-weight indexing. The **2025 Cohort** is smaller (**197 active accounts**) but shows a higher average SIP ticket size of **Rs. 13,505.21**, with **SBI Small Cap Fund** emerging as their top preference. This suggests that newer retail inflows are skewed toward higher-risk, small-cap segments.

### 3. SIP Continuity & Mandate Friction
Out of 1,362 investors who have registered at least 6 consecutive SIP transactions, a massive **97.80% (1,332 accounts)** are flagged as **'at-risk'** with an average inter-transaction gap exceeding 35 days. Under ideal conditions, a monthly SIP transaction occurs every 28 to 31 days. Gaps averaging 35 to 45 days point to frequent skipped cycles, late payments, or bank mandate failures. This behavioral friction highlights the need for fund houses to implement automatic reminders and automated backup mandates to prevent account churn.

### 4. Portfolio Concentration & Diversification Patterns (HHI)
The **Herfindahl-Hirschman Index (HHI)** calculation reveals a substantial divergence in how fund managers allocate capital:
- The **Axis Bluechip Fund (AMFI: 119092)** has the most concentrated holdings, with an HHI score of **2,064.48** across only 10 stock holdings, exposing it to substantial stock-specific risk.
- The **SBI Small Cap Fund (AMFI: 119598)** is highly diversified, with an HHI of **1,073.49** spread across 12 holdings, mitigating stock-specific risk.
An HHI above 1,500 indicates a highly concentrated allocation strategy, characteristic of active large-cap managers seeking alpha, whereas HHI scores below 1,200 highlight a diversified structure typical of small-cap funds managing liquidity requirements.

### 5. Sharpe Ratio-based Asset Recommendations
Evaluating Sharpe ratios within risk grade categories provides a solid framework for matching individual risk profiles with efficient portfolios. Under the **Moderate** risk appetite mapping (which joins `Moderate` and `Moderately High` risk categories), the recommendation utility identifies the **HDFC Top 100 Fund** (Sharpe: 1.06), **Mirae Asset Large Cap Fund** (Sharpe: 1.06), and **ICICI Prudential Bluechip Fund** (Sharpe: 1.03) as the most efficient risk-adjusted portfolios. This quantitative allocation process helps investors maximize returns per unit of volatility.